# 🏦 对抗学习量化系统 - 全自动云端管线 v2.4

**全自动执行4个阶段：**
1. **Phase 1** — 数据获取（baostock下载A股日K线）
2. **Phase 2** — 生成器对抗训练（C-TimeGAN）
3. **Phase 3** — 庄散对抗竞技场（多组合进化）
4. **Phase 4** — 结果解读与股票推荐

**断点续传**：每个Phase完成后自动保存checkpoint，Colab断开后重连可从断点恢复。

**运行时间估计**：Phase 1 ~5min | Phase 2 ~2h | Phase 3 ~4h | Phase 4 ~10min

## Cell 0: 环境安装与目录准备

In [ ]:
import os, sys, time, json, warnings, shutil, base64, urllib.request
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
print(f'🕐 开始时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'📍 Python: {sys.version}')

# 安装依赖
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q baostock pandas scikit-learn lightgbm matplotlib

import torch
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ NumPy: {np.__version__}')

if torch.cuda.is_available():
    print(f'🚀 GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    print('💻 仅CPU模式')
    DEVICE = 'cpu'

# 创建目录结构
BASE_DIR = Path('/content/AdversarialLearning')
DIRS = {
    'base': BASE_DIR,
    'dotpy': BASE_DIR / 'dotpy',
    'stockdata': BASE_DIR / 'stockdata',
    'adv_data': BASE_DIR / 'adversarial_data',
    'adv_model': BASE_DIR / 'adversarial_model',
    'results': BASE_DIR / 'stockresults',
    'checkpoint': BASE_DIR / 'adversarial_model' / 'checkpoint',
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# 进度追踪
PROGRESS_FILE = BASE_DIR / 'pipeline_progress.json'

def save_progress(phase, status, detail=''):
    progress = {}
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            progress = json.load(f)
    progress[phase] = {
        'status': status,
        'detail': detail,
        'time': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)
    print(f'📝 进度保存: Phase {phase} -> {status}')

def load_progress():
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {}

print('✅ 目录结构创建完成')
print(f'📁 Base: {BASE_DIR}')

## Cell 1: 下载最新脚本（从GitHub）

In [ ]:
# ===== Cell 1: 从GitHub下载脚本 =====
GITHUB_REPO = 'Scilogos/scilogos.github.io'
GITHUB_BRANCH = 'main'
GITHUB_DIR = 'cai/dotpy'
GITHUB_TOKEN = 'YOUR_GITHUB_TOKEN'

SCRIPTS_TO_DOWNLOAD = [
    'adversarial_env.py',
    'market_generator.py',
    'stock_config.py',
    'stock_data_manager.py',
    'stock_interpreter.py',
    'run_pipeline.py',
    'feedback_processor.py',
    'pipeline_monitor.py',
]

headers = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

print('📥 从GitHub下载最新脚本...')
for fname in SCRIPTS_TO_DOWNLOAD:
    url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{GITHUB_DIR}/{fname}?ref={GITHUB_BRANCH}'
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read())
            content = base64.b64decode(data['content'])
            dest = DIRS['dotpy'] / fname
            with open(dest, 'wb') as f:
                f.write(content)
            print(f'  OK {fname} ({len(content)} bytes)')
    except Exception as e:
        print(f'  FAIL {fname}: {e}')

# 覆写 stock_config.py 为云端自适应版本
print('\n📝 生成云端配置 stock_config.py ...')

config_py = '''"""
stock_config.py - Cloud Auto-Adapt Config
"""
import os, sys, json, logging, hashlib, platform
import numpy as np
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple
from pathlib import Path
from datetime import datetime

def _detect_env():
    if "COLAB_GPU" in os.environ or os.path.exists("/content"):
        return "colab"
    elif platform.system() == "Windows":
        return "windows"
    return "linux"

ENV = _detect_env()

if ENV == "colab":
    BASE_DIR = Path("/content/AdversarialLearning")
elif ENV == "windows":
    BASE_DIR = Path(r"C:\\Users\\HUAWEI\\Desktop\\Adversarial Learning")
else:
    BASE_DIR = Path("/app/data/cloud_arena/workspace")

SCRIPT_DIR = BASE_DIR / "dotpy"
DATA_DIR = BASE_DIR / "stockdata"
ADV_DATA_DIR = BASE_DIR / "adversarial_data"
ADV_MODEL_DIR = BASE_DIR / "adversarial_model"
RESULTS_DIR = BASE_DIR / "stockresults"
FEEDBACK_FILE = SCRIPT_DIR / "feedback.txt"
PYMANAGER_FILE = SCRIPT_DIR / "pymanager.txt"

for d in [SCRIPT_DIR, DATA_DIR, ADV_DATA_DIR, ADV_MODEL_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PYTHON_EXE = sys.executable

BS_FIELDS_DAILY = [
    "date", "code", "open", "high", "low", "close",
    "preclose", "volume", "amount", "adjustflag",
    "turn", "tradestatus", "pctChg", "isST"
]
DAILY_START = "2021-01-01"
DAILY_END = "2026-06-25"
FOCUS_POOL_SIZE = 200

@dataclass
class GeneratorConfig:
    hidden_dim: int = 64
    num_layers: int = 3
    seq_len: int = 30
    feature_dim: int = 6
    condition_levels: int = 3
    phase_a_epochs: int = 80
    phase_b_epochs: int = 80
    phase_c_epochs: int = 150
    batch_size: int = 64
    learning_rate: float = 1e-3
    dtw_threshold: float = 2.0
    pearson_threshold: float = 0.85
    ks_pvalue: float = 0.05
    w930_940: float = 5.0
    w940_950: float = 2.0
    w_other: float = 1.0
    max_similar: int = 3

@dataclass
class AdversarialConfig:
    num_episodes: int = 500
    episode_length: int = 240
    initial_price: float = 10.0
    dealer_capital_ratio: float = 0.30
    dealer_info_manip_prob: float = 0.05
    retailer_ratio: float = 0.55
    retailer_monthly_salary: float = 10000.0
    retailer_types: List[str] = field(default_factory=lambda: [
        "herd", "value", "technical", "leader", "passive"
    ])
    hotmoney_ratio: float = 0.15
    hotmoney_momentum_thresh: float = 0.03
    lamarck_rate: float = 0.1
    darwin_rate: float = 0.05
    evolution_interval: int = 50
    anti_degenerate_thresh: float = 0.1
    min_strategy_diversity: float = 0.3

@dataclass
class InterpreterConfig:
    stat_window: int = 20
    game_depth: int = 5
    historical_topk: int = 10
    max_drawdown: float = 0.15
    perm_samples: int = 1000
    significance: float = 0.05
    signal_conf_thresh: float = 0.6

@dataclass
class FeedbackConfig:
    feedback_file: str = str(FEEDBACK_FILE)
    results_dir: str = str(RESULTS_DIR)
    model_dir: str = str(ADV_MODEL_DIR)
    calibration_window: int = 20
    signal_accuracy_thresh: float = 0.55
    max_position_adjust: float = 0.1
    reward_shaping_weight: float = 0.3

def normalize_ohlcv(arr):
    mn = arr.min(axis=0, keepdims=True)
    mx = arr.max(axis=0, keepdims=True)
    rng = mx - mn
    rng[rng == 0] = 1.0
    return (arr - mn) / rng

def denormalize_ohlcv(arr, mn, mx):
    rng = mx - mn
    rng[rng == 0] = 1.0
    return arr * rng + mn

def pct_change(close):
    out = np.zeros_like(close)
    out[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-8)
    return out

def save_config(cfg, path):
    with open(path, "w") as f:
        json.dump(asdict(cfg), f, ensure_ascii=False, indent=2)

def load_config(cfg_cls, path):
    with open(path) as f:
        return cfg_cls(**json.load(f))

def setup_logger(name, log_file=None, level=logging.INFO):
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(level)
    fmt = logging.Formatter(
        "[%(asctime)s][%(name)s][%(levelname)s] %(message)s",
        datefmt="%H:%M:%S"
    )
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    if log_file:
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    return logger
'''

with open(DIRS['dotpy'] / 'stock_config.py', 'w') as f:
    f.write(config_py)

print('OK stock_config.py (cloud auto-adapt) generated')

# 验证导入
sys.path.insert(0, str(DIRS['dotpy']))
import importlib
if 'stock_config' in sys.modules:
    importlib.reload(sys.modules['stock_config'])
import stock_config
print(f'OK config loaded: ENV={stock_config.ENV}, BASE={stock_config.BASE_DIR}')
save_progress('setup', 'done', 'All scripts downloaded and config generated')

## Cell 2: Phase 1 — 数据获取
从Baostock下载A股日K线数据。聚焦200只代表性股票。

⏱️ 预计时间: 5-10分钟

In [ ]:
# ===== Cell 2: Phase 1 数据获取 =====
import baostock as bs

progress = load_progress()
if progress.get('phase1', {}).get('status') == 'done':
    print('Phase 1 already done, skipping')
else:
    print('Phase 1: Data Download')
    print('=' * 50)

    lg = bs.login()
    print(f'Baostock login: {lg.error_msg}')

    # 获取所有A股列表
    print('\nGetting stock list...')
    stock_df = bs.query_all_stock(day=datetime.now().strftime('%Y-%m-%d')).get_data()
    stock_codes = stock_df[stock_df['type'] == '1']['code'].tolist()
    print(f'Total A-shares: {len(stock_codes)}')

    # 选择200只代表性股票
    np.random.seed(42)
    priority_codes = []

    large_caps = [c for c in stock_codes if c.startswith('sh.60')]
    priority_codes.extend(large_caps[:50])

    shenzhen_main = [c for c in stock_codes if c.startswith('sz.00')]
    priority_codes.extend(np.random.choice(shenzhen_main, min(50, len(shenzhen_main)), replace=False).tolist())

    chinext = [c for c in stock_codes if c.startswith('sz.30')]
    priority_codes.extend(np.random.choice(chinext, min(50, len(chinext)), replace=False).tolist())

    star = [c for c in stock_codes if c.startswith('sh.68')]
    priority_codes.extend(np.random.choice(star, min(50, len(star)), replace=False).tolist())

    focus_codes = list(set(priority_codes))[:200]
    print(f'Focus pool: {len(focus_codes)} stocks')

    # 下载日K线
    print(f'\nDownloading daily K-line ({stock_config.DAILY_START} ~ {stock_config.DAILY_END})...')
    success = 0
    failed = 0

    for i, code in enumerate(focus_codes):
        try:
            rs = bs.query_history_k_data_plus(
                code,
                'date,code,open,high,low,close,volume,amount,pctChg,turn,tradestatus,isST',
                start_date=stock_config.DAILY_START,
                end_date=stock_config.DAILY_END,
                frequency='d',
                adjustflag='2'
            )
            rows = []
            while rs.next():
                rows.append(rs.get_row_data())

            if rows:
                df = pd.DataFrame(rows, columns=rs.fields)
                for col in ['open','high','low','close','volume','amount','pctChg','turn']:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                safe_name = code.replace('.', '_')
                df.to_csv(DIRS['stockdata'] / f'{safe_name}.csv', index=False)
                success += 1
            else:
                failed += 1
        except Exception as e:
            failed += 1

        if (i+1) % 20 == 0:
            print(f'  Progress: {i+1}/{len(focus_codes)} (ok={success}, fail={failed})')

    bs.logout()
    print(f'\nPhase 1 done: {success} ok, {failed} failed')

    with open(DIRS['stockdata'] / 'focus_pool.json', 'w') as f:
        json.dump(focus_codes, f, indent=2)

    save_progress('phase1', 'done', f'success={success}, failed={failed}, codes={len(focus_codes)}')

# 显示数据概况
csv_files = list(DIRS['stockdata'].glob('sh_*.csv')) + list(DIRS['stockdata'].glob('sz_*.csv'))
print(f'\nData files: {len(csv_files)}')
for f in csv_files[:3]:
    df = pd.read_csv(f)
    print(f'  {f.name}: {len(df)} rows, {df["date"].iloc[0]}~{df["date"].iloc[-1]}')

## Cell 3: Phase 2 — 生成器对抗训练
训练 C-TimeGAN 生成器学习A股数据分布特征。

⏱️ 预计时间: 1-3小时

In [ ]:
# ===== Cell 3: Phase 2 生成器训练 =====
sys.path.insert(0, str(DIRS['dotpy']))

progress = load_progress()
if progress.get('phase2', {}).get('status') == 'done':
    print('Phase 2 already done, skipping')
else:
    print('Phase 2: Generator Training')
    print('=' * 50)

    from market_generator import MarketDataGenerator, GeneratorConfig

    gen = MarketDataGenerator()

    # Phase A
    print('\nPhase A: Autoencoder training...')
    try:
        result_a = gen.train_phase_a(
            data_dir=str(DIRS['stockdata']),
            epochs=80,
            batch_size=64
        )
        print(f'  Phase A loss: {result_a.get("final_loss", "N/A")}')
    except Exception as e:
        print(f'  Phase A error: {e}')
        result_a = {'status': 'simplified'}

    # Phase B
    print('\nPhase B: Supervised training...')
    try:
        result_b = gen.train_phase_b(
            data_dir=str(DIRS['stockdata']),
            epochs=80,
            batch_size=64
        )
        print(f'  Phase B loss: {result_b.get("final_loss", "N/A")}')
    except Exception as e:
        print(f'  Phase B error: {e}')
        result_b = {'status': 'simplified'}

    # Phase C
    print('\nPhase C: Adversarial training...')
    try:
        result_c = gen.train_phase_c(
            data_dir=str(DIRS['stockdata']),
            epochs=150,
            batch_size=64
        )
        print(f'  Phase C D={result_c.get("d_loss", "N/A")} G={result_c.get("g_loss", "N/A")}')
    except Exception as e:
        print(f'  Phase C error: {e}')
        result_c = {'status': 'simplified'}

    gen.save_model(str(DIRS['adv_model'] / 'generator.pt'))

    # 生成验证数据
    print('\nGenerating validation data...')
    try:
        fake_data = gen.generate(n_samples=1000)
        np.save(DIRS['adv_data'] / 'generated_1000.npy', fake_data)
        print(f'  Shape: {fake_data.shape}')
        print(f'  Range: [{fake_data.min():.3f}, {fake_data.max():.3f}]')
    except Exception as e:
        print(f'  Generate error: {e}, using GBM fallback')
        # GBM fallback
        n_samples, seq_len, feat_dim = 1000, 30, 6
        prices = np.cumsum(np.random.randn(n_samples, seq_len, feat_dim) * 0.02, axis=1) + 10.0
        np.save(DIRS['adv_data'] / 'generated_1000.npy', prices)

    save_progress('phase2', 'done', f'A={result_a.get("status","ok")} B={result_b.get("status","ok")} C={result_c.get("status","ok")}')
    print('\nPhase 2 done')

## Cell 4: Phase 3 — 庄散对抗竞技场 ⭐核心
多组合进化竞技场，8种庄散组合多元化选拔。

⏱️ 预计时间: 2-6小时

In [ ]:
# ===== Cell 4: Phase 3 庄散对抗 =====
sys.path.insert(0, str(DIRS['dotpy']))

progress = load_progress()
if progress.get('phase3', {}).get('status') == 'done':
    print('Phase 3 already done, skipping')
else:
    print('Phase 3: Adversarial Arena')
    print('=' * 50)

    from adversarial_env import AdversarialArena, NeuralFiringMechanism, MarketPhase

    # 确保有数据可用
    gen_data_path = DIRS['adv_data'] / 'generated_1000.npy'
    if not gen_data_path.exists():
        print('No generated data, creating from real data...')
        import glob
        csv_files = glob.glob(str(DIRS['stockdata'] / '*.csv'))
        all_prices = []
        for f in csv_files[:10]:
            df = pd.read_csv(f)
            if len(df) > 30:
                prices = df[['open','high','low','close','volume','pctChg']].values
                all_prices.append(prices)
        if all_prices:
            benchmark = np.concatenate(all_prices, axis=0)
            np.save(DIRS['adv_data'] / 'price_benchmark.npy', benchmark)
            print(f'  Benchmark: {benchmark.shape}')

    print('\nCreating arena...')
    arena = AdversarialArena(
        data_path=str(DIRS['adv_data']),
        model_path=str(DIRS['adv_model']),
    )

    print('\nStarting adversarial training...')
    print(f'  Episodes: 500')
    print(f'  Episode Length: 240 ticks')
    print(f'  Neural firing threshold: 0.6321 (63.21%)')

    try:
        result = arena.train(
            num_episodes=500,
            evolve=True,
            checkpoint_every=50,
            checkpoint_dir=str(DIRS['checkpoint'])
        )

        with open(DIRS['results'] / 'arena_result.json', 'w') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)

        print(f'\nResults:')
        print(f'  Dealer avg reward: {result.get("dealer_avg_reward", "N/A")}')
        print(f'  Retailer avg reward: {result.get("retailer_avg_reward", "N/A")}')
        print(f'  HotMoney avg reward: {result.get("hotmoney_avg_reward", "N/A")}')

    except Exception as e:
        print(f'\nTraining error: {e}')
        import traceback
        traceback.print_exc()
        result = {'status': 'error', 'error': str(e)}

    arena.save_model(str(DIRS['adv_model'] / 'adversarial_model.pt'))
    save_progress('phase3', 'done', str(result.get('status', 'unknown')))
    print('\nPhase 3 done')

## Cell 5: Phase 4 — 结果解读与股票推荐
基于对抗训练结果，生成市场分析报告和个股推荐。

⏱️ 预计时间: 5-10分钟

In [ ]:
# ===== Cell 5: Phase 4 结果解读 =====
sys.path.insert(0, str(DIRS['dotpy']))

progress = load_progress()
if progress.get('phase4', {}).get('status') == 'done':
    print('Phase 4 already done, skipping')
else:
    print('Phase 4: Result Interpretation')
    print('=' * 50)

    try:
        from stock_interpreter import StockInterpreter

        interpreter = StockInterpreter(
            results_dir=str(DIRS['results']),
            model_dir=str(DIRS['adv_model']),
            data_dir=str(DIRS['stockdata'])
        )

        print('\nGenerating market analysis report...')
        report = interpreter.generate_full_report()

        report_path = DIRS['results'] / 'market_analysis_report.md'
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report)

        print(report)

        with open(DIRS['results'] / 'interpretation.json', 'w') as f:
            json.dump({'report': report, 'time': datetime.now().isoformat()}, f,
                     ensure_ascii=False, indent=2)

        save_progress('phase4', 'done', f'report={len(report)} chars')

    except Exception as e:
        print(f'Interpretation error: {e}')
        import traceback
        traceback.print_exc()
        # 生成基础报告
        print('\nGenerating basic report...')
        arena_file = DIRS['results'] / 'arena_result.json'
        if arena_file.exists():
            with open(arena_file) as f:
                arena_data = json.load(f)
            report = f'# Arena Results\n\n{json.dumps(arena_data, indent=2, ensure_ascii=False, default=str)}'
        else:
            report = '# Pipeline completed\n\nNo arena results found.'
        with open(DIRS['results'] / 'market_analysis_report.md', 'w') as f:
            f.write(report)
        print(report)
        save_progress('phase4', 'done', 'basic report')

print('\nPhase 4 done')

## Cell 6: 保存结果到 Google Drive
将所有结果持久化到Google Drive，防止Colab断开后丢失。

In [ ]:
# ===== Cell 6: 保存到Google Drive =====
print('Saving results to Google Drive')
print('=' * 50)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_BASE = Path('/content/drive/MyDrive/AdversarialLearning')
    GDRIVE_BASE.mkdir(parents=True, exist_ok=True)

    for subdir in ['stockresults', 'adversarial_model', 'adversarial_data']:
        src = BASE_DIR / subdir
        dst = GDRIVE_BASE / subdir
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'  OK {subdir}/ synced to Drive')

    shutil.copy2(PROGRESS_FILE, GDRIVE_BASE / 'pipeline_progress.json')
    print(f'\nAll results saved to: {GDRIVE_BASE}')

except ImportError:
    print('Not in Colab, skipping Drive save')
    print(f'Results at: {BASE_DIR}')
except Exception as e:
    print(f'Drive save failed: {e}')
    print(f'Results still at: {BASE_DIR}')

# 最终汇总
print('\n' + '=' * 50)
print('Pipeline complete!')
print('=' * 50)
print(f'\nResults: {BASE_DIR / "stockresults"}')

final_progress = load_progress()
for phase, info in final_progress.items():
    icon = 'OK' if info['status'] == 'done' else 'FAIL'
    print(f'  [{icon}] {phase}: {info["status"]} ({info["time"]})')

report_file = DIRS['results'] / 'market_analysis_report.md'
if report_file.exists():
    print(f'\nReport preview:')
    print('-' * 40)
    with open(report_file) as f:
        for line in f.read().split('\n')[:30]:
            print(f'  {line}')

print(f'\nEnd time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Cell 7: 断点续传
Colab断开后重连，运行此Cell可从上次中断处继续。

In [ ]:
# ===== Cell 7: 断点续传 =====
print('Checking resume point...')
print('=' * 50)

# 先重新执行 Cell 0 (环境安装) 和 Cell 1 (下载脚本)
# 然后检查进度
progress = load_progress()

if not progress:
    print('No progress file found, please start from Cell 0')
else:
    print('Current progress:')
    for phase, info in progress.items():
        icon = 'OK' if info['status'] == 'done' else '...'
        print(f'  [{icon}] {phase}: {info["status"]} ({info["time"]})')

    phases = ['phase1', 'phase2', 'phase3', 'phase4']
    for p in phases:
        if progress.get(p, {}).get('status') != 'done':
            print(f'\nResume from {p}...')
            print(f'   Run the corresponding Phase Cell ({phases.index(p)+2})')
            break
    else:
        print('\nAll phases done! Run Cell 5 (interpretation) or Cell 6 (save to Drive)')

# 显示最近日志
log_files = list((DIRS['adv_model']).glob('*.log'))
if log_files:
    latest_log = max(log_files, key=lambda f: f.stat().st_mtime)
    print(f'\nLatest log: {latest_log.name}')
    with open(latest_log) as f:
        lines = f.readlines()
    for line in lines[-20:]:
        print(f'  {line.rstrip()}')